# Chapter 8: Building the receptionist

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [Building the receptionist](https://books.sugiv.fyi/voice_applications/building-the-receptionist.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## The receptionist in one loop

## A bench of callers who talk back

In [ ]:
import json
import re
import sys

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, "sim")
from bench_calls import CALLS  # the sixteen calls
from receptionist import ticket_id
from scoring import normalize  # Chapter 4

BY_ID = {c["id"]: c for c in CALLS}
print(json.dumps({k: BY_ID["c04"][k] for k in ["lines", "knows", "tools", "say"]}, indent=1, ensure_ascii=False))

In [ ]:
ACTIONS = {"book_service": "booked", "take_message": "taken"}


def same(kind, got, want):
    got = str(got)
    if kind == "ticket":
        return ticket_id(got) == want
    if kind == "phone":
        return re.sub(r"\D", "", got) == want
    if kind in ("name", "caller_name"):
        return normalize(got) == normalize(want)
    return got.strip().lower() == want.lower()


def tool_calls(turns):
    """Every tool call in a call; a confirmed two-phase action counts as the booking or message it saved."""
    out = []
    for i, t in enumerate(turns):
        for s in t["steps"]:
            for c in s["calls"]:
                r = c.get("result") or {}
                if c.get("name") == "confirm_action" and r.get("booked"):
                    c = {"name": "book_service", "arguments": {k: r[k] for k in ("name", "bike", "service", "day")},
                         "result": r}
                elif c.get("name") == "confirm_action" and r.get("taken"):
                    c = {"name": "take_message", "result": r,
                         "arguments": {"staff": r["for"], "caller_name": r["from"], "phone": r["phone"]}}
                elif c.get("name") in ACTIONS and r.get("pending"):
                    continue  # a proposal, not an action
                out.append((i, c))
    return out


def judge(call, turns):
    calls = tool_calls(turns)
    replies = " ".join(t["reply"] for t in turns)
    done = lambda c: c.get("name") in ACTIONS and c.get("result", {}).get(ACTIONS[c["name"]])
    matches = lambda c, tool, want: c.get("name") == tool and all(
        same(k, c.get("arguments", {}).get(k, ""), v) for k, v in want.items())
    needed = all(any(matches(c, tool, want) and (tool not in ACTIONS or done(c)) for _, c in calls)
                 for tool, want in call["tools"])
    wrong = sum(done(c) and not any(matches(c, t, w) for t, w in call["tools"]) for _, c in calls)
    facts = all(re.search(p, replies, re.I) for p in call["say"])
    invented = [p for p in call["never"] if re.search(p, replies, 0 if "[A-Z]" in p else re.I)]
    acted = [i for i, c in calls if done(c)]
    return {"handled": needed and not wrong and facts and not invented, "tools": needed, "wrong": wrong,
            "facts": facts, "invented": bool(invented), "acted": len(acted),
            "waited": sum(turns[i]["line"].lower().startswith(("yes", "yeah")) for i in acted),
            "corrections": sum("_fix_" in t["clip"] for t in turns)}


RUNS = ["1.7b-snap-guarded", "4b-snap-guarded", "8b-snap-guarded", "4b-text-guarded", "4b-asr-guarded",
        "4b-snap-plain", "4b-snap-twophase", "8b-snap-twophase", "4b-snap-code", "8b-snap-code"]
talk = {r: json.load(open(f"data/ch8/talk_{r}.json")) for r in RUNS}
scores = {r: {cid: judge(BY_ID[cid], turns) for cid, turns in talk[r]["calls"].items()} for r in RUNS}


def summary(r):
    s = scores[r].values()
    return {"handled": sum(x["handled"] for x in s), "wrong": sum(x["wrong"] for x in s),
            "invented": sum(x["invented"] for x in s), "acted": sum(x["acted"] for x in s),
            "waited": sum(x["waited"] for x in s), "corrections": sum(x["corrections"] for x in s)}


print(f"{'run':20s} handled  wrong actions  made up  waited for yes  corrections")
for r in RUNS:
    s = summary(r)
    print(f"{r:20s} {s['handled']:4d}/16 {s['wrong']:10d} {s['invented']:9d} {s['waited']:9d} of {s['acted']:<3d}"
          f" {s['corrections']:9d}")

## Which brain?

In [ ]:
def show(run, cid, width=150):
    for t in talk[run]["calls"][cid]:
        print(f"CALLER  (heard as) {t['heard']}")
        for s in t["steps"]:
            for c in s["calls"]:
                args = ", ".join(f"{k}={v!r}" for k, v in (c.get("arguments") or {}).items())
                result = c.get("result") or {}
                outcome = ("saved" if result.get("booked") or result.get("taken") else
                           "pending" if result.get("pending") else json.dumps(result)[:60])
                print(f"   TOOL {c.get('name')}({args}) -> {outcome}" + ("   [by code]" if c.get("by") else ""))
        print(f"RECEPTIONIST {t['reply'][:width]}")


show("4b-snap-guarded", "c04")

## A rule in the prompt is a request

In [ ]:
show("4b-snap-guarded", "c14")

In [ ]:
show("4b-snap-guarded", "c15")

## A rule in code is a rule

In [ ]:
for r in ["4b-snap-guarded", "4b-snap-twophase", "4b-snap-code", "8b-snap-guarded", "8b-snap-twophase",
          "8b-snap-code"]:
    s = summary(r)
    print(f"{r:20s} handled {s['handled']:2d}/16   wrong actions {s['wrong']}   waited for yes "
          f"{s['waited']} of {s['acted']}")

In [ ]:
show("4b-snap-code", "c04")

In [ ]:
show("8b-snap-code", "c16")

## Where the waiting goes

In [ ]:
hear = json.load(open("data/ch8/hear.json"))
speak = json.load(open("data/ch8/speak.json"))
rows = []
for cid, turns in talk["4b-snap-guarded"]["calls"].items():
    for t in turns:
        if not t["steps"]:
            continue  # nothing was heard, so nothing was said
        steps, h = t["steps"], hear[t["clip"]]
        first = re.split(r"(?<=[.!?])\s", t["reply"].strip())[0]
        rows.append({"endpoint": h["endpoint_wait"], "asr": h["asr_s"],
                     "tools": sum(s["timing"]["total"] for s in steps[:-1]),
                     "sentence": steps[-1]["timing"]["first_sentence"] or steps[-1]["timing"]["total"],
                     "whole_reply": steps[-1]["timing"]["total"], "tts": speak[first]["pocket_cpu"],
                     "tool_turn": len(steps) > 1, "first_token": steps[0]["timing"]["first_token"]})
lat = {k: np.array([r[k] for r in rows]) for k in rows[0]}
waits = {
    "wait for the whole reply, then speak": lat["endpoint"] + lat["asr"] + lat["tools"] + lat["whole_reply"] + lat["tts"],
    "speak the first sentence as soon as it's written": lat["endpoint"] + lat["asr"] + lat["tools"] + lat["sentence"] + lat["tts"],
    "...and say 'one moment' while tools run": np.where(lat["tool_turn"], lat["endpoint"] + lat["asr"] + lat["first_token"],
                                                    lat["endpoint"] + lat["asr"] + lat["tools"] + lat["sentence"] + lat["tts"]),
}
print(f"{len(rows)} turns, {int(lat['tool_turn'].sum())} of them with tool calls\n")
print(f"{'stage':28s} median   90th pct")
for k, label in [("endpoint", "end-of-turn wait"), ("asr", "Parakeet"), ("tools", "tool calls (when any)"),
                 ("sentence", "model, first sentence"), ("tts", "Pocket TTS, first audio")]:
    v = lat[k][lat["tool_turn"]] if k == "tools" else lat[k]
    print(f"{label:28s} {np.median(v):5.2f} s  {np.percentile(v, 90):5.2f} s")
print()
for label, v in waits.items():
    print(f"{label:50s} median {np.median(v):.2f} s, 90th percentile {np.percentile(v, 90):.2f} s")

fig, ax = plt.subplots(figsize=(7, 2.6))
parts = [("end-of-turn wait", "endpoint", "#9e9e9e"), ("Parakeet", "asr", "#4a7fb0"),
         ("tool calls", "tools", "#e0a040"), ("model", "sentence", "#c0504d"), ("TTS", "tts", "#5a9e5a")]
left = 0
for label, k, color in parts:
    width = float(np.median(lat[k][lat["tool_turn"]])) if k == "tools" else float(np.median(lat[k]))
    ax.barh(0, width, left=left, color=color, label=label)
    left += width
ax.set(yticks=[], xlabel="seconds after the caller stops talking (medians; tool calls on tool turns)",
       title="Where the caller's wait goes")
ax.legend(ncol=5, fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.45))
plt.tight_layout()
plt.show()

## Rerun the models

Everything above used the saved results. The cells below rerun the models that produced them, on a GPU (in Colab:
Runtime, then Change runtime type, then T4 GPU). Each step builds the software it needs in its own environment the
first time, which takes a few minutes, then runs. The first cell below sets up a two-item check that everything
works, on a scratch copy of the folder, so the saved results stay as they are. Delete its lines for a full rerun,
which overwrites the saved results with yours. On GPUs without bfloat16, such as the T4, the scripts use float16
automatically.

In [ ]:
# A two-item check that every step runs, on a scratch copy so the saved results stay untouched.
# Delete these lines for a full rerun, which overwrites the saved results with your own.
import os, shutil, tempfile
os.environ["VOTF_QUICK"] = "2"
scratch = os.path.join(tempfile.gettempdir(), "votf-quick")
shutil.copytree(".", scratch, dirs_exist_ok=True)
os.chdir(scratch)

In [ ]:
!bash tools/stack.sh tts
!~/.votf/envs/tts/bin/python sim/capture_ch8.py --part voice

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch8.py --part hear

Qwen3-8B doesn't fit a 16 GB T4 in fp16; its runs fail there.

In [ ]:
!bash tools/stack.sh hf
!~/.votf/envs/hf/bin/python sim/capture_ch8.py --part talk

In [ ]:
!bash tools/stack.sh tts
!~/.votf/envs/tts/bin/python sim/capture_ch8.py --part speak